<a href="https://colab.research.google.com/github/karquiza/bikeshare-toronto-demand/blob/main/01_load_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# MAI 103 Group Assignment 1 – Milestone 3 (EDA)
# Notebook 01: Load and Clean Raw Trip Files
# ==============================================================================

# 1. SETUP & CONFIGURATION
import os
import glob
import re
import pandas as pd
import numpy as np

# Folder structure setup matching your environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/DataSets"
else:
    DATA = "../data"

RAW_DIR = f"{DATA}/raw"
CLEAN_DIR = f"{DATA}/clean"
PROFILE_DIR = f"{DATA}/profile"

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(PROFILE_DIR, exist_ok=True)

# Discover and sort all raw files chronologically
raw_files = sorted(glob.glob(f"{RAW_DIR}/*/*.csv"))

# Mapping dictionary for 2023 schema variants to 2024 standardized names
RENAME_2023 = {
    "Trip Id": "Trip_Id",
    "Trip  Duration": "Trip_Duration", # Captures the double whitespace
    "Start Station Id": "Start_Station_Id",
    "Start Time": "Start_Time",
    "End Station Id": "End_Station_Id",
    "End Time": "End_Time",
}

# Targeted feature matrix columns to drop names, user type, and bike model
FINAL_COLUMNS = [
    "Trip_Id", "Trip_Duration", "Start_Time", "End_Time",
    "Start_Station_Id", "End_Station_Id",
    "flag_short", "flag_long", "flag_no_end", "is_valid",
    "is_new_start_station", "is_new_end_station"
]

# ==============================================================================
# 2. PHASE 1: DISCOVER STATION NETWORK CHRONOLOGY (Anti-Data Leakage)
# ==============================================================================
print("Phase 1: Analyzing historical station appearance timeline...")
station_first_seen = {}

for path in raw_files:
    # Safely extract year and month components from path string
    file_match = re.search(r"(20\d\d)[-_](\d\d)", os.path.basename(path))
    year_month_key = f"{file_match.group(1)}-{file_match.group(2)}"

    # Load minimal structure to deduce structural names
    sample_df = pd.read_csv(path, nrows=5)
    sample_df.columns = [c.replace("\ufeff", "") for c in sample_df.columns]

    start_col = "Start Station Id" if "Start Station Id" in sample_df.columns else "Start_Station_Id"
    end_col = "End Station Id" if "End Station Id" in sample_df.columns else "End_Station_Id"

    # Extract unique indices for the network baseline matrix
    stations_chunk = pd.read_csv(path, usecols=[start_col, end_col])
    unique_start = stations_chunk[start_col].dropna().astype("Int64").unique()
    unique_end = stations_chunk[end_col].dropna().astype("Int64").unique()
    all_current_stations = set(unique_start).union(set(unique_end))

    for station_id in all_current_stations:
        if station_id not in station_first_seen:
            station_first_seen[station_id] = year_month_key

print(f"Network discovery complete. Documented {len(station_first_seen)} unique stations across history.")

# ==============================================================================
# 3. PHASE 2: STANDARDIZATION & CLEANING LOGIC LOOP
# ==============================================================================
print("\nPhase 2: Executing normalization transforms and validation matrix rules...")
summary_metrics = []

for path in raw_files:
    file_match = re.search(r"(20\d\d)[-_](\d\d)", os.path.basename(path))
    year = file_match.group(1)
    month = file_match.group(2)
    year_month_key = f"{year}-{month}"

    print(f"Processing batch: {year_month_key}...")

    # Load raw table chunk
    df = pd.read_csv(path, low_memory=False)
    total_raw_rows = len(df)

    # Clean hidden bytes from structural elements
    df.columns = [c.replace("\ufeff", "") for c in df.columns]

    # Enforce 2024 standardized target schema mapping
    if "Trip Id" in df.columns:
        df = df.rename(columns=RENAME_2023)
        # Parse 2023 dates without breaking on missing seconds
        df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="%m/%d/%Y %H:%M", errors="coerce")
        df["End_Time"] = pd.to_datetime(df["End_Time"], format="%m/%d/%Y %H:%M", errors="coerce")
    else:
        df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
        df["End_Time"] = pd.to_datetime(df["End_Time"], errors="coerce")

    # Cast identifiers to Pandas Int64 array extension to preserve structural integers with nulls safely
    df["Trip_Id"] = df["Trip_Id"].astype("Int64")
    df["Start_Station_Id"] = df["Start_Station_Id"].astype("Int64")
    df["End_Station_Id"] = df["End_Station_Id"].astype("Int64")

    # R4: Safe Drop Exact Duplicates on unique Trip_Id index
    df = df.drop_duplicates(subset=["Trip_Id"])
    dropped_duplicates = total_raw_rows - len(df)

    # Apply Row-by-Row Flag Matrices
    # R1: Short Trips (< 1 Minute / 60 Seconds)
    df["flag_short"] = df["Trip_Duration"] < 60

    # R2: Long Trips (> 6 Hours / 21,600 Seconds)
    df["flag_long"] = df["Trip_Duration"] > 21600

    # R3: Missing End Station ID
    df["flag_no_end"] = df["End_Station_Id"].isna()

    # Structural validity index column evaluations
    df["is_valid"] = ~(df["flag_short"] | df["flag_long"] | df["flag_no_end"])

    # Flag infrastructure infrastructure updates without looking forward
    df["is_new_start_station"] = df["Start_Station_Id"].map(station_first_seen) == year_month_key
    df["is_new_end_station"] = df["End_Station_Id"].map(station_first_seen) == year_month_key

    # Append accounting balances to summary metric array
    summary_metrics.append({
        "Year": int(year),
        "Month": int(month),
        "Year_Month": year_month_key,
        "Total_Raw_Trips": total_raw_rows,
        "Dropped_Duplicates": dropped_duplicates,
        "Flag_Short": df["flag_short"].sum(),
        "Flag_Long": df["flag_long"].sum(),
        "Flag_No_End": df["flag_no_end"].sum(),
        "Valid_Trips": df["is_valid"].sum()
    })

    # Prune attributes down to defined targeted tracking columns and serialize
    df_clean = df[FINAL_COLUMNS]
    clean_out_name = f"clean_ridership_{year}_{month}.csv"
    df_clean.to_csv(os.path.join(CLEAN_DIR, clean_out_name), index=False)

# ==============================================================================
# 4. METRIC BALANCES AND VERIFICATION OUTPUTS
# ==============================================================================
df_summary = pd.DataFrame(summary_metrics)
summary_out_path = os.path.join(PROFILE_DIR, "cleaning_summary_counts.csv")
df_summary.to_csv(summary_out_path, index=False)

print("\nProcessing completed successfully!")
print(f"Normalized files saved to targets: {CLEAN_DIR}")
print(f"Global profile tracking matrix saved to target: {summary_out_path}")

# Display brief sample snapshot out of the final summary run matrix
print("\nSnapshot of Data Audit Log Matrix:")
print(df_summary.head().to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase 1: Analyzing historical station appearance timeline...
Network discovery complete. Documented 1114 unique stations across history.

Phase 2: Executing normalization transforms and validation matrix rules...
Processing batch: 2023-01...
Processing batch: 2023-02...
Processing batch: 2023-03...
Processing batch: 2023-04...
Processing batch: 2023-05...
Processing batch: 2023-06...
Processing batch: 2023-07...
Processing batch: 2023-08...
Processing batch: 2023-09...
Processing batch: 2023-10...
Processing batch: 2023-11...
Processing batch: 2023-12...
Processing batch: 2024-01...
Processing batch: 2024-02...
Processing batch: 2024-03...
Processing batch: 2024-04...
Processing batch: 2024-05...
Processing batch: 2024-06...
Processing batch: 2024-07...
Processing batch: 2024-08...
Processing batch: 2024-09...
Processing batch: 2024-10...
Processing batch: 20